# Conectando con la BBDD

In [ ]:
%pip install oracledb pandas -q

import pandas as pd
import oracledb


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Conectar a Oracle SIN tnsnames.ora (evita DPY-4027) y verificar variables

%pip install -q oracledb python-dotenv

import os, oracledb
from dotenv import load_dotenv

load_dotenv(override=True)

# Requeridas en .env
req = ["DB_USER","DB_PASS","ORA_HOST","ORA_PORT","ORA_SERVICE"]
vals = {k: os.getenv(k) for k in req}
print("Vars:", vals)

missing = [k for k,v in vals.items() if not v]
if missing:
    raise SystemExit(f"Faltan en .env: {missing}\nEjemplo:\nDB_HOST=mi-host\nDB_PORT=1521\nDB_SERVICE=mi_servicio")

# Forzar a no usar TNS
os.environ.pop("TNS_ADMIN", None)

# DSN programático
dsn = oracledb.makedsn(host=vals["ORA_HOST"], port=int(vals["ORA_PORT"]), service_name=vals["ORA_SERVICE"])

# Conexión
conn = oracledb.connect(user=vals["DB_USER"], password=vals["DB_PASS"], dsn=dsn)

# Prueba
with conn.cursor() as cur:
    cur.execute("select 1 from dual")
    print("OK Oracle:", cur.fetchone())
conn.close()


In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import oracledb

# Cargar variables desde .env
load_dotenv()  # lee .env en el directorio actual
REQUIRED = ["DB_USER", "DB_PASS", "DB_DSN"]
missing = [k for k in REQUIRED if not os.getenv(k)]
if missing:
    raise RuntimeError(f"Faltan variables en .env: {missing}")

DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_DSN  = os.getenv("DB_DSN")   # ej: //host:1521/servicio

# Carga de datos
conn = oracledb.connect(user=DB_USER, password=DB_PASS, dsn=DB_DSN)
sql = """
SELECT * FROM PMATOWNER.PMAT_PREDICTION
"""
df = pd.read_sql(sql, conn)
conn.close()

print("Dimensiones:", df.shape)
df.head(15)



DatabaseError: DPY-4027: no configuration directory specified

In [21]:
import pandas as pd, sys
sys.path.insert(0, 'src')
from dotenv import load_dotenv; load_dotenv()
from oracle_connector import OracleConnector

conn = OracleConnector()
df = pd.DataFrame(conn.read_table('PMAT_PREDICTION'))
df['TARGET_REAL'] = pd.to_numeric(df['TARGET_REAL'], errors='coerce')

print(f"Total registros         : {len(df):,}")
print(f"Con TARGET_REAL         : {df['TARGET_REAL'].notna().sum():,}")
print(f"Sin TARGET_REAL (null)  : {df['TARGET_REAL'].isna().sum():,}")
print(f"TARGET_REAL = 1 (matr.) : {(df['TARGET_REAL']==1).sum():,}")
print(f"TARGET_REAL = 0 (no m.) : {(df['TARGET_REAL']==0).sum():,}")

ModuleNotFoundError: No module named 'oracle_connector'

In [22]:
# Asegura import de src/oracle_connector.py sin errores de ruta/entorno

import sys, os, pathlib, importlib

# 1) Mostrar cwd y contenido de src
print("CWD:", os.getcwd())
print("Existe src?:", pathlib.Path("src").exists())
print("oracle_connector.py en src?:", pathlib.Path("src/oracle_connector.py").exists())

# 2) Añadir src al sys.path si no está
src_path = str(pathlib.Path("src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print("src en sys.path?:", src_path in sys.path)

# 3) Invalidar caché de imports y probar
importlib.invalidate_caches()
try:
    from oracle_connector import OracleConnector
    print("Import OK: OracleConnector")
except Exception as e:
    print("Fallo import:", repr(e))
    # pista extra: listar módulos homónimos
    import pkgutil
    print("Módulos que empiezan por 'oracle':", [m.name for m in pkgutil.iter_modules() if m.name.startswith("oracle")])
    raise

# 4) Si sigues en otro directorio, ajusta cwd al repo (opcional)
# os.chdir("/ruta/a/tu/repo"); sys.path.insert(0, str(pathlib.Path("src").resolve()))


CWD: c:\Users\malmendrosv\UNAV\notebooks
Existe src?: False
oracle_connector.py en src?: False
src en sys.path?: True
Fallo import: ModuleNotFoundError("No module named 'oracle_connector'")
Módulos que empiezan por 'oracle': ['oracledb']


ModuleNotFoundError: No module named 'oracle_connector'

In [18]:
# Conexión Oracle sin tnsnames.ora (evita DPY-4027) + verificación de variables

%pip install -q oracledb python-dotenv

import os, oracledb
from dotenv import load_dotenv

load_dotenv()
req = ["DB_USER","DB_PASS","ORA_HOST","ORA_PORT","ORA_SERVICE"]
missing = [k for k in req if not os.getenv(k)]
if missing:
    raise SystemExit(f"Faltan variables en .env: {missing}\nEjemplo:\nDB_HOST=host\nDB_PORT=1521\nDB_SERVICE=servicio")

DB_USER=os.getenv("DB_USER")
DB_PASS=os.getenv("DB_PASS")
ORA_HOST=os.getenv("ORA_HOST")
ORA_PORT=int(os.getenv("ORA_PORT", "1521"))
ORA_SERVICE=os.getenv("ORA_SERVICE")

# Desactiva cualquier TNS_ADMIN para esta sesión
os.environ.pop("TNS_ADMIN", None)

# Construye DSN programáticamente
dsn = oracledb.makedsn(host=ORA_HOST, port=ORA_PORT, service_name=ORA_SERVICE)

# Conexión directa
conn = oracledb.connect(user=DB_USER, password=DB_PASS, dsn=dsn)

# Smoke test
with conn.cursor() as cur:
    cur.execute("select 1 from dual")
    print("OK Oracle:", cur.fetchone())

conn.close()


Note: you may need to restart the kernel to use updated packages.
OK Oracle: (1,)



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
# Cargar .env y verificar variables requeridas
%pip install -q python-dotenv oracledb

import os, oracledb
from dotenv import load_dotenv

load_dotenv(override=True)  # asegúrate de leer el .env del cwd

req = ["DB_USER","DB_PASS","ORA_HOST","ORA_PORT","ORA_SERVICE"]
vals = {k: os.getenv(k) for k in req}
print("Vars leídas:", vals)

missing = [k for k,v in vals.items() if not v]
if missing:
    raise SystemExit(f"Faltan en .env: {missing}\nEjemplo:\nDB_HOST=host\nDB_PORT=1521\nDB_SERVICE=servicio")

# Forzar conexión sin TNS
os.environ.pop("TNS_ADMIN", None)
dsn = oracledb.makedsn(host=vals["ORA_HOST"], port=int(vals["ORA_PORT"]), service_name=vals["ORA_SERVICE"])
conn = oracledb.connect(user=vals["DB_USER"], password=vals["DB_PASS"], dsn=dsn)

with conn.cursor() as cur:
    cur.execute("select 1 from dual")
    print("OK Oracle:", cur.fetchone())
conn.close()


Note: you may need to restart the kernel to use updated packages.
Vars leídas: {'DB_USER': 'PMAT_BTCH[PMATOWNER]', 'DB_PASS': 'w6IT%_)M>&', 'ORA_HOST': 'racdb-pre.si.unav.es', 'ORA_PORT': '1521', 'ORA_SERVICE': 'UNSIDPRE.UNAV'}
OK Oracle: (1,)



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
